# 05 - Inferencia con Feature Store

Este notebook demuestra las ventajas de usar **Feature Store** en un pipeline
de Computer Vision:

1. **Lookup automatico de features**: Enriquecer predicciones YOLO con features
   precomputadas del Feature Store sin duplicar logica de feature engineering
2. **score_batch**: Usar `FeatureEngineeringClient.score_batch()` para inferencia
   batch con lookups automaticos
3. **Consistencia**: Las mismas features usadas en entrenamiento se reutilizan
   en inferencia sin riesgo de training-serving skew
4. **Linaje en Unity Catalog**: Trazabilidad completa de datos a modelo

**Prerequisitos:** Ejecutar notebooks 01-04

In [0]:
%pip install numpy==1.26.4 ultralytics databricks-feature-engineering -q
%pip uninstall -y opencv-python -q
%pip install --force-reinstall --no-deps opencv-python-headless==4.10.0.84 -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import mlflow
from pyspark.sql import functions as F

# Configurar directorio de ultralytics antes de importar (evita PermissionError en serverless)
os.environ["YOLO_CONFIG_DIR"] = "/tmp/ultralytics_cfg"
os.makedirs("/tmp/ultralytics_cfg", exist_ok=True)

from ultralytics import YOLO
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, ArrayType
)
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

# -- Unity Catalog --
CATALOG = "jgworkspaceclassic_catalog"
SCHEMA = "infra_demo"

GOLD_IMAGE_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_gold_image_features"
GOLD_ANNOTATION_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_gold_annotation_features"
PREDICTIONS_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_predictions"
ENRICHED_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_predictions_enriched"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/computer_vision"
MODEL_WEIGHTS = os.path.join(VOLUME_PATH, "models", "best_v2.pt")

fe = FeatureEngineeringClient()

print(f"Model weights: {MODEL_WEIGHTS}")
print(f"Feature tables:")
print(f"  Images:      {GOLD_IMAGE_TABLE}")
print(f"  Annotations: {GOLD_ANNOTATION_TABLE}")

/databricks/python/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Model weights: /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/models/best_v2.pt
Feature tables:
  Images:      jgworkspaceclassic_catalog.infra_demo.cv_v2_gold_image_features
  Annotations: jgworkspaceclassic_catalog.infra_demo.cv_v2_gold_annotation_features


## Ventaja 1: Feature Store como fuente unica de verdad

En lugar de recalcular features durante inferencia, las consultamos del
Feature Store. Esto garantiza **consistencia** entre entrenamiento e inferencia.

In [0]:
# Ver las features que tenemos disponibles
print("=" * 60)
print("FEATURES DISPONIBLES EN FEATURE STORE")
print("=" * 60)

print(f"\n--- {GOLD_IMAGE_TABLE} ---")
df_img = spark.table(GOLD_IMAGE_TABLE)
print(f"  Registros: {df_img.count()}")
print(f"  Columnas:  {df_img.columns}")

print(f"\n--- {GOLD_ANNOTATION_TABLE} ---")
df_ann = spark.table(GOLD_ANNOTATION_TABLE)
print(f"  Registros: {df_ann.count()}")
print(f"  Columnas:  {df_ann.columns}")

FEATURES DISPONIBLES EN FEATURE STORE

--- jgworkspaceclassic_catalog.infra_demo.cv_v2_gold_image_features ---
  Registros: 175
  Columnas:  ['image_id', 'image_path', 'split', 'num_objects', 'num_captain', 'num_gohan', 'num_distinct_classes', 'avg_bbox_area', 'min_bbox_area', 'max_bbox_area', 'std_bbox_area', 'total_bbox_coverage', 'avg_aspect_ratio', 'avg_polygon_complexity', 'max_polygon_complexity', 'image_size_bytes', 'avg_centroid_x', 'avg_centroid_y', 'classes_present', 'has_both_classes', 'dominant_class', 'class_balance_ratio', 'object_density', 'computed_at']

--- jgworkspaceclassic_catalog.infra_demo.cv_v2_gold_annotation_features ---
  Registros: 189
  Columnas:  ['image_id', 'object_index', 'split', 'class_id', 'class_name', 'bbox_x_min', 'bbox_y_min', 'bbox_x_max', 'bbox_y_max', 'bbox_width', 'bbox_height', 'bbox_area', 'bbox_perimeter', 'bbox_aspect_ratio', 'centroid_x', 'centroid_y', 'distance_to_center', 'num_polygon_points', 'size_category', 'position_quadrant', 'shap

In [0]:
model = YOLO(MODEL_WEIGHTS)
print(f"Modelo cargado: {MODEL_WEIGHTS}")
print(f"  Clases: {model.names}")

Modelo cargado: /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/models/best_v2.pt
  Clases: {0: 'Captain', 1: 'Gohan'}


In [0]:
# Obtener imagenes del split test desde Feature Store
df_test_images = (
    spark.table(GOLD_IMAGE_TABLE)
    .filter(F.col("split") == "test")
    .select("image_id", "image_path")
)

test_image_paths = [row.image_path for row in df_test_images.collect()]
print(f"Imagenes de test: {len(test_image_paths)}")

# Ejecutar inferencia YOLO en todas las imagenes de test
prediction_rows = []
for img_path in test_image_paths:
    if not os.path.exists(img_path):
        continue

    results = model.predict(img_path, conf=0.25, verbose=False)

    image_id = os.path.splitext(os.path.basename(img_path))[0]

    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            # Imagen sin detecciones
            prediction_rows.append((
                image_id, img_path, -1, "none", 0.0,
                0.0, 0.0, 0.0, 0.0, 0
            ))
            continue

        for obj_idx, box in enumerate(r.boxes):
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            conf = float(box.conf[0])
            # Coordenadas normalizadas
            x1, y1, x2, y2 = box.xyxyn[0].tolist()

            prediction_rows.append((
                image_id, img_path, cls_id, cls_name, conf,
                float(x1), float(y1), float(x2), float(y2), obj_idx
            ))

print(f"Total predicciones: {len(prediction_rows)}")

Imagenes de test: 7
Total predicciones: 10


In [0]:
pred_schema = StructType([
    StructField("image_id", StringType(), False),
    StructField("image_path", StringType(), True),
    StructField("pred_class_id", IntegerType(), False),
    StructField("pred_class_name", StringType(), False),
    StructField("confidence", FloatType(), False),
    StructField("pred_bbox_x_min", FloatType(), False),
    StructField("pred_bbox_y_min", FloatType(), False),
    StructField("pred_bbox_x_max", FloatType(), False),
    StructField("pred_bbox_y_max", FloatType(), False),
    StructField("pred_object_index", IntegerType(), False),
])

df_predictions = spark.createDataFrame(prediction_rows, schema=pred_schema)
df_predictions = df_predictions.withColumn("predicted_at", F.current_timestamp())

# Guardar predicciones raw
df_predictions.write.mode("overwrite").saveAsTable(PREDICTIONS_TABLE)
print(f"Predicciones guardadas: {PREDICTIONS_TABLE}")
print(f"Total: {df_predictions.count()}")
display(df_predictions.limit(10))

Predicciones guardadas: jgworkspaceclassic_catalog.infra_demo.cv_v2_predictions
Total: 10


image_id,image_path,pred_class_id,pred_class_name,confidence,pred_bbox_x_min,pred_bbox_y_min,pred_bbox_x_max,pred_bbox_y_max,pred_object_index,predicted_at
IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674.jpg,1,Gohan,0.9974044,0.003605783,0.3280613,0.3979238,0.77148,0,2026-04-23T01:36:34.739Z
IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674.jpg,0,Captain,0.99205923,0.53416646,0.38931054,0.92594826,0.7814756,1,2026-04-23T01:36:34.739Z
IMG_20260421_072911977_jpg.rf.a185ba4b826c92c00a6c804f23d67f0c,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072911977_jpg.rf.a185ba4b826c92c00a6c804f23d67f0c.jpg,1,Gohan,0.94359547,0.16705735,0.18395305,0.80450344,0.8253996,0,2026-04-23T01:36:34.739Z
IMG_20260421_072403468_jpg.rf.bd2a936b8dc51ed34ca02bf2a21d63eb,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072403468_jpg.rf.bd2a936b8dc51ed34ca02bf2a21d63eb.jpg,0,Captain,0.854779,0.20627776,0.22073603,0.7220344,0.82901824,0,2026-04-23T01:36:34.739Z
IMG_20260421_072943027_jpg.rf.edaae16fa079c65f78a9f260a81b2b69,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072943027_jpg.rf.edaae16fa079c65f78a9f260a81b2b69.jpg,1,Gohan,0.30477953,0.1682642,0.18740831,0.7568783,0.8065912,0,2026-04-23T01:36:34.739Z
IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d.jpg,1,Gohan,0.7757346,0.17839196,0.14838581,0.70465994,0.7184245,0,2026-04-23T01:36:34.739Z
IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d.jpg,1,Gohan,0.39283136,0.17838125,0.14976645,0.7035076,0.7240612,1,2026-04-23T01:36:34.739Z
IMG_20260421_072835266_jpg.rf.3eb576f3e87b1c0b522f6d516d48042e,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072835266_jpg.rf.3eb576f3e87b1c0b522f6d516d48042e.jpg,1,Gohan,0.9340697,0.12170084,0.1806057,0.7641008,0.88076496,0,2026-04-23T01:36:34.739Z
IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7.jpg,1,Gohan,0.664206,0.12741457,0.20166765,0.67591125,0.7668709,0,2026-04-23T01:36:34.739Z
IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7.jpg,1,Gohan,0.40506235,0.12784381,0.20048545,0.6754427,0.7621361,1,2026-04-23T01:36:34.739Z


## Ventaja 2: Feature Lookup automatico

Usamos `FeatureLookup` para enriquecer las predicciones con features
precomputadas. No necesitamos recalcular nada - el Feature Store hace
el JOIN automaticamente usando `image_id` como clave.

In [0]:
# Definir lookups desde Feature Store
feature_lookups = [
    FeatureLookup(
        table_name=GOLD_IMAGE_TABLE,
        lookup_key=["image_id"],
        feature_names=[
            "num_objects",
            "num_captain",
            "num_gohan",
            "avg_bbox_area",
            "total_bbox_coverage",
            "dominant_class",
            "has_both_classes",
            "object_density",
            "avg_aspect_ratio",
        ],
    ),
]

# Crear training set (tambien sirve para inferencia) con lookups
df_preds = spark.table(PREDICTIONS_TABLE)

# create_training_set enriquece automaticamente con features del Feature Store
training_set = fe.create_training_set(
    df=df_preds,
    feature_lookups=feature_lookups,
    label=None,
)
df_enriched = training_set.load_df()

print("Predicciones enriquecidas con Feature Store:")
display(df_enriched.limit(10))

Predicciones enriquecidas con Feature Store:


image_id,image_path,pred_class_id,pred_class_name,confidence,pred_bbox_x_min,pred_bbox_y_min,pred_bbox_x_max,pred_bbox_y_max,pred_object_index,predicted_at,num_objects,num_captain,num_gohan,avg_bbox_area,total_bbox_coverage,dominant_class,has_both_classes,object_density,avg_aspect_ratio
IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674.jpg,1,Gohan,0.9974044,0.003605783,0.3280613,0.3979238,0.77148,0,2026-04-23T01:36:33.644Z,2,1,1,0.16526124626398087,0.33052249252796173,balanced,true,0.33052249252796173,0.9572232278765291
IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674.jpg,0,Captain,0.99205923,0.53416646,0.38931054,0.92594826,0.7814756,1,2026-04-23T01:36:33.644Z,2,1,1,0.16526124626398087,0.33052249252796173,balanced,true,0.33052249252796173,0.9572232278765291
IMG_20260421_072911977_jpg.rf.a185ba4b826c92c00a6c804f23d67f0c,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072911977_jpg.rf.a185ba4b826c92c00a6c804f23d67f0c.jpg,1,Gohan,0.94359547,0.16705735,0.18395305,0.80450344,0.8253996,0,2026-04-23T01:36:33.644Z,1,0,1,0.3993603587150574,0.3993603587150574,Gohan,false,0.3993603587150574,0.9683697571931403
IMG_20260421_072403468_jpg.rf.bd2a936b8dc51ed34ca02bf2a21d63eb,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072403468_jpg.rf.bd2a936b8dc51ed34ca02bf2a21d63eb.jpg,0,Captain,0.854779,0.20627776,0.22073603,0.7220344,0.82901824,0,2026-04-23T01:36:33.644Z,1,1,0,0.31687501072883606,0.31687501072883606,Captain,false,0.31687501072883606,0.8802082784887839
IMG_20260421_072943027_jpg.rf.edaae16fa079c65f78a9f260a81b2b69,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072943027_jpg.rf.edaae16fa079c65f78a9f260a81b2b69.jpg,1,Gohan,0.30477953,0.1682642,0.18740831,0.7568783,0.8065912,0,2026-04-23T01:36:33.644Z,1,0,1,0.35584717988967896,0.35584717988967896,Gohan,false,0.35584717988967896,0.9341772538197192
IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d.jpg,1,Gohan,0.7757346,0.17839196,0.14838581,0.70465994,0.7184245,0,2026-04-23T01:36:33.644Z,1,0,1,0.3000781238079071,0.3000781238079071,Gohan,false,0.3000781238079071,0.907608755932865
IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d.jpg,1,Gohan,0.39283136,0.17838125,0.14976645,0.7035076,0.7240612,1,2026-04-23T01:36:33.644Z,1,0,1,0.3000781238079071,0.3000781238079071,Gohan,false,0.3000781238079071,0.907608755932865
IMG_20260421_072835266_jpg.rf.3eb576f3e87b1c0b522f6d516d48042e,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072835266_jpg.rf.3eb576f3e87b1c0b522f6d516d48042e.jpg,1,Gohan,0.9340697,0.12170084,0.1806057,0.7641008,0.88076496,0,2026-04-23T01:36:33.644Z,1,0,1,0.4476684629917145,0.4476684629917145,Gohan,false,0.4476684629917145,0.8857142689463856
IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7.jpg,1,Gohan,0.664206,0.12741457,0.20166765,0.67591125,0.7668709,0,2026-04-23T01:36:33.644Z,1,0,1,0.3057495355606079,0.3057495355606079,Gohan,false,0.3057495355606079,0.9504131432388228
IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7.jpg,1,Gohan,0

### Alternativa: Enriquecer sin score_batch (JOIN directo)

Si `score_batch` con `model_uri=None` no esta soportado, podemos hacer
el lookup manualmente. Esto sigue usando el Feature Store como fuente
de verdad, pero con un JOIN explicito.

In [0]:
df_preds = spark.table(PREDICTIONS_TABLE)
df_img_features = spark.table(GOLD_IMAGE_TABLE)

# JOIN con features de imagen
df_enriched = (
    df_preds
    .join(
        df_img_features.select(
            "image_id",
            "num_objects", "num_captain", "num_gohan",
            "avg_bbox_area", "total_bbox_coverage",
            "dominant_class", "has_both_classes",
            "object_density", "avg_aspect_ratio",
            "avg_polygon_complexity",
            "num_distinct_classes",
        ),
        on="image_id",
        how="left",
    )
    # Agregar features derivadas de la prediccion vs ground truth
    .withColumn("pred_bbox_area",
        (F.col("pred_bbox_x_max") - F.col("pred_bbox_x_min")) *
        (F.col("pred_bbox_y_max") - F.col("pred_bbox_y_min"))
    )
    .withColumn("pred_matches_dominant",
        F.col("pred_class_name") == F.col("dominant_class"))
)

# Guardar
df_enriched.write.mode("overwrite").saveAsTable(ENRICHED_TABLE)
print(f"Tabla enriquecida guardada: {ENRICHED_TABLE}")
display(df_enriched.limit(10))

Tabla enriquecida guardada: jgworkspaceclassic_catalog.infra_demo.cv_v2_predictions_enriched


image_id,image_path,pred_class_id,pred_class_name,confidence,pred_bbox_x_min,pred_bbox_y_min,pred_bbox_x_max,pred_bbox_y_max,pred_object_index,predicted_at,num_objects,num_captain,num_gohan,avg_bbox_area,total_bbox_coverage,dominant_class,has_both_classes,object_density,avg_aspect_ratio,avg_polygon_complexity,num_distinct_classes,pred_bbox_area,pred_matches_dominant
IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674.jpg,1,Gohan,0.9974044,0.003605783,0.3280613,0.3979238,0.77148,0,2026-04-23T01:36:33.644Z,2,1,1,0.16526124626398087,0.33052249252796173,balanced,true,0.33052249252796173,0.9572232278765291,154.5,2,0.17484799,false
IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674.jpg,0,Captain,0.99205923,0.53416646,0.38931054,0.92594826,0.7814756,1,2026-04-23T01:36:33.644Z,2,1,1,0.16526124626398087,0.33052249252796173,balanced,true,0.33052249252796173,0.9572232278765291,154.5,2,0.15364313,false
IMG_20260421_072911977_jpg.rf.a185ba4b826c92c00a6c804f23d67f0c,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072911977_jpg.rf.a185ba4b826c92c00a6c804f23d67f0c.jpg,1,Gohan,0.94359547,0.16705735,0.18395305,0.80450344,0.8253996,0,2026-04-23T01:36:33.644Z,1,0,1,0.3993603587150574,0.3993603587150574,Gohan,false,0.3993603587150574,0.9683697571931403,185.0,1,0.4088876,true
IMG_20260421_072403468_jpg.rf.bd2a936b8dc51ed34ca02bf2a21d63eb,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072403468_jpg.rf.bd2a936b8dc51ed34ca02bf2a21d63eb.jpg,0,Captain,0.854779,0.20627776,0.22073603,0.7220344,0.82901824,0,2026-04-23T01:36:33.644Z,1,1,0,0.31687501072883606,0.31687501072883606,Captain,false,0.31687501072883606,0.8802082784887839,120.0,1,0.31372556,true
IMG_20260421_072943027_jpg.rf.edaae16fa079c65f78a9f260a81b2b69,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072943027_jpg.rf.edaae16fa079c65f78a9f260a81b2b69.jpg,1,Gohan,0.30477953,0.1682642,0.18740831,0.7568783,0.8065912,0,2026-04-23T01:36:33.644Z,1,0,1,0.35584717988967896,0.35584717988967896,Gohan,false,0.35584717988967896,0.9341772538197192,164.0,1,0.36445978,true
IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d.jpg,1,Gohan,0.7757346,0.17839196,0.14838581,0.70465994,0.7184245,0,2026-04-23T01:36:33.644Z,1,0,1,0.3000781238079071,0.3000781238079071,Gohan,false,0.3000781238079071,0.907608755932865,166.0,1,0.29999313,true
IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d.jpg,1,Gohan,0.39283136,0.17838125,0.14976645,0.7035076,0.7240612,1,2026-04-23T01:36:33.644Z,1,0,1,0.3000781238079071,0.3000781238079071,Gohan,false,0.3000781238079071,0.907608755932865,166.0,1,0.3015773,true
IMG_20260421_072835266_jpg.rf.3eb576f3e87b1c0b522f6d516d48042e,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072835266_jpg.rf.3eb576f3e87b1c0b522f6d516d48042e.jpg,1,Gohan,0.9340697,0.12170084,0.1806057,0.7641008,0.88076496,0,2026-04-23T01:36:33.644Z,1,0,1,0.4476684629917145,0.4476684629917145,Gohan,false,0.4476684629917145,0.8857142689463856,200.0,1,0.44978228,true
IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7,/Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/test/images/IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7.jpg,1,Gohan,0.664206,0.12741457,0.20166765,0.67591125,0.7668709,0,2026-04-23T01:36:33.644Z,1,0,1,0.3057495355606079,0.30574953

## Ventaja 3: Analisis enriquecido gracias al Feature Store

Con las features del Feature Store ya adjuntas, podemos hacer analisis
avanzados sin recalcular nada.

In [0]:
# Como varia la confianza del modelo segun la densidad de objetos?
df_analysis = spark.table(ENRICHED_TABLE).filter(F.col("pred_class_id") >= 0)

print("Confianza promedio por densidad de objetos en la imagen:")
display(
    df_analysis
    .withColumn("density_bucket",
        F.when(F.col("object_density") < 0.05, "baja (<5%)")
         .when(F.col("object_density") < 0.15, "media (5-15%)")
         .otherwise("alta (>15%)")
    )
    .groupBy("density_bucket")
    .agg(
        F.count("*").alias("predicciones"),
        F.avg("confidence").cast("decimal(4,3)").alias("avg_confidence"),
        F.min("confidence").cast("decimal(4,3)").alias("min_confidence"),
    )
    .orderBy("density_bucket")
)

Confianza promedio por densidad de objetos en la imagen:


density_bucket,predicciones,avg_confidence,min_confidence
alta (>15%),10,0.726,0.305


In [0]:
# Comparar predicciones con ground truth del Feature Store
print("Predicciones vs Ground Truth (Feature Store):")
display(
    df_analysis
    .groupBy("image_id")
    .agg(
        F.count("*").alias("predicciones"),
        F.first("num_objects").alias("gt_objects"),
        F.avg("confidence").cast("decimal(4,3)").alias("avg_confidence"),
        F.first("dominant_class").alias("gt_dominant_class"),
        F.first("has_both_classes").alias("gt_has_both"),
    )
    .withColumn("diff_objects",
        F.col("predicciones") - F.col("gt_objects"))
    .orderBy("image_id")
)

Predicciones vs Ground Truth (Feature Store):


image_id,predicciones,gt_objects,avg_confidence,gt_dominant_class,gt_has_both,diff_objects
IMG_20260421_072403468_jpg.rf.bd2a936b8dc51ed34ca02bf2a21d63eb,1,1,0.855,Captain,false,0
IMG_20260421_072835266_jpg.rf.3eb576f3e87b1c0b522f6d516d48042e,1,1,0.934,Gohan,false,0
IMG_20260421_072911977_jpg.rf.a185ba4b826c92c00a6c804f23d67f0c,1,1,0.944,Gohan,false,0
IMG_20260421_072943027_jpg.rf.edaae16fa079c65f78a9f260a81b2b69,1,1,0.305,Gohan,false,0
IMG_20260421_072950352_jpg.rf.3c4e863ceed05394a8df66750a9911e7,2,1,0.535,Gohan,false,1
IMG_20260421_073008531_jpg.rf.0ce412b4200a645bbc0d34a987be571d,2,1,0.584,Gohan,false,1
IMG_20260421_073100036_jpg.rf.b2cd33a005c0165199bfcb7551aa7674,2,2,0.995,balanced,true,0


In [0]:
# Analisis cruzado: prediccion vs features de anotacion
df_ann = spark.table(GOLD_ANNOTATION_TABLE).filter(F.col("split") == "test")

print("Distribucion de predicciones por clase y cuadrante (ground truth):")
display(
    df_ann
    .groupBy("class_name", "size_category", "position_quadrant")
    .agg(F.count("*").alias("ground_truth_count"))
    .orderBy("class_name", "size_category")
)

Distribucion de predicciones por clase y cuadrante (ground truth):


class_name,size_category,position_quadrant,ground_truth_count
Captain,very_large,bottom_right,1
Captain,very_large,bottom_left,1
Gohan,very_large,bottom_left,3
Gohan,very_large,top_left,3


## Ventaja 4: Linaje completo en Unity Catalog

Gracias a usar Feature Store + MLflow + Unity Catalog, tenemos trazabilidad
completa:

```
Roboflow Dataset
    |
    v
Bronze (cv_v2_bronze) -- datos crudos
    |
    v
Silver (cv_v2_silver) -- datos validados
    |
    v
Gold Feature Store
  |- cv_v2_gold_image_features (features por imagen)
  |- cv_v2_gold_annotation_features (features por anotacion)
    |
    v
Modelo YOLO (registrado en UC)
  |- Entrenado con datos del pipeline
  |- Metricas en MLflow
  |- Parametros vinculados a Feature Store
    |
    v
Predicciones enriquecidas (cv_v2_predictions_enriched)
```

Todo visible en el **Catalog Explorer** de Databricks.

In [0]:
print("=" * 70)
print("RESUMEN COMPLETO DEL PIPELINE v2")
print("=" * 70)

tables = [
    (f"{CATALOG}.{SCHEMA}.cv_v2_bronze", "Bronze - Datos crudos"),
    (f"{CATALOG}.{SCHEMA}.cv_v2_silver", "Silver - Datos validados"),
    (GOLD_IMAGE_TABLE, "Gold - Image Features (FS)"),
    (GOLD_ANNOTATION_TABLE, "Gold - Annotation Features (FS)"),
    (PREDICTIONS_TABLE, "Predicciones raw"),
    (ENRICHED_TABLE, "Predicciones + Feature Store"),
]

for table_name, desc in tables:
    try:
        count = spark.table(table_name).count()
        print(f"  {desc:45s} | {count:>6} rows | {table_name}")
    except Exception as e:
        print(f"  {desc:45s} | ERROR  | {table_name}: {e}")

RESUMEN COMPLETO DEL PIPELINE v2
  Bronze - Datos crudos                         |    189 rows | jgworkspaceclassic_catalog.infra_demo.cv_v2_bronze
  Silver - Datos validados                      |    189 rows | jgworkspaceclassic_catalog.infra_demo.cv_v2_silver
  Gold - Image Features (FS)                    |    175 rows | jgworkspaceclassic_catalog.infra_demo.cv_v2_gold_image_features
  Gold - Annotation Features (FS)               |    189 rows | jgworkspaceclassic_catalog.infra_demo.cv_v2_gold_annotation_features
  Predicciones raw                              |     10 rows | jgworkspaceclassic_catalog.infra_demo.cv_v2_predictions
  Predicciones + Feature Store                  |     10 rows | jgworkspaceclassic_catalog.infra_demo.cv_v2_predictions_enriched


## Resumen de beneficios del Feature Store

| Beneficio | Sin Feature Store | Con Feature Store |
| --- | --- | --- |
| **Consistencia** | Recalcular features en inferencia (riesgo de skew) | Features identicas a entrenamiento |
| **Eficiencia** | Recalcular cada vez | Precomputado, solo lookup |
| **Linaje** | Manual, fragil | Automatico en Unity Catalog |
| **Reutilizacion** | Copiar codigo de features | Compartir tabla de features entre equipos |
| **Gobernanza** | Sin control de acceso | Permisos de Unity Catalog |
| **Versionado** | Archivos sueltos | Delta tables con time travel |